In [ ]:
load_ext jupyter_black

In [ ]:
from copy import deepcopy
import numpy as np
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.multivariate.multivariate_ols import _MultivariateOLS
import statsmodels.formula.api as smf
import matplotlib.patches as mpatches
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
import pyfixest as pf

from dim_erasure import binarize_df

In [ ]:
def scale_data(df):
    # from https://github.com/CJTAYL/elastic_net_medium/blob/main/data_cleaning.py
    """
    Function to scale numerical data

    Parameters
    ----------
    df: dataframe
        Dataframe containing true labels of groups (clusters)

    Returns
    ----------
    df_scaled: dataframe
        Dataframe containing scaled values of numeric variables
    """
    numeric_columns = df.select_dtypes(include=["float64", "int"]).columns
    categorical_columns = df.select_dtypes(exclude=["float64", "int"]).columns
    ct = ColumnTransformer(
        [("scale", StandardScaler(), numeric_columns)], remainder="passthrough"
    )

    # Fit and transform the data
    df_scaled_array = ct.fit_transform(df)

    # ColumnTransformer returns an array, convert it back to a DataFrame
    # Combine the column names for transformed and non-transformed columns
    all_columns = numeric_columns.tolist() + categorical_columns.tolist()
    df_scaled = pd.DataFrame(df_scaled_array, columns=all_columns, index=df.index)

    return df_scaled

In [ ]:
demographics = {
    "prism": [
        "age",
        "gender",
        "employment_status",
        "education",
        "marital_status",
        "english_proficiency",
        "religion",
        "ethnicity",
        "birth_region",
        "reside_region",
        "lm_familiarity",
    ],
    "personamem": [
        "gender",
        "age",
        "ethnicity",
    ],
    "cad_en": [
        "annotator_age",
        "annotator_gender",
        "annotator_education_level",
        "annotator_political",
        "annotator_ethnicity",
    ],
}

domains = ["legal", "salary", "medical", "benefits", "political"]

In [ ]:
def get_answers(model):
    questions = pd.read_pickle(f"data/{model}_questions.gz")
    questions_correct_answers = dict(zip(questions.q_id, questions.correct_answer))
    questions_baseline_answers = questions[["q_id", "baseline_answer", "domain"]]

    questions_baseline_answers = questions_baseline_answers.loc[
        ~questions_baseline_answers["q_id"].isin([f"q_{i}" for i in range(61)])
    ].reset_index(drop=True)

    domain_qid_map = {
        domain: questions.loc[questions["domain"] == domain, "q_id"].tolist()
        for domain in domains
    }

    questions_baseline_answers.loc[
        questions_baseline_answers["domain"] == "salary",
        "baseline_answer",
    ] = (
        questions_baseline_answers.loc[
            questions_baseline_answers["domain"] == "salary",
            "baseline_answer",
        ]
        .str.replace(",", "")
        .str.extract(r"^[^\d]*(\d+)", expand=False)
        .astype(float)
    )

    questions_baseline_answers = pd.Series(
        questions_baseline_answers.baseline_answer.values,
        index=questions_baseline_answers.q_id,
    ).to_dict()
    return domain_qid_map, questions_correct_answers, questions_baseline_answers

In [ ]:
model = "Qwen3.6-27B"  # ["gemma-3-12b-it", "Llama-3.1-8B-Instruct","Qwen3.6-27B"]
dataset = "cad_en"  # ["cad_en", "personamem","prism",]
domain = "salary"  # ["benefits", "legal", "medical", "political", "salary"]

In [ ]:
domain_qid_map, questions_correct_answers, questions_baseline_answers = get_answers(
    model
)

all_cols = deepcopy(demographics[dataset])
if model == "Llama-3.1-8B-Instruct":
    df = pd.read_pickle(f"behavior/{model}_{dataset}_answers.gz")
else:
    df = pd.read_pickle(f"behavior/{model}_{dataset}_{domain}_answers.gz")
if domain != "salary":
    for c in domain_qid_map[domain]:
        df[c] = 1 * (df[c].str.lower() == questions_correct_answers[c])
else:
    for c in domain_qid_map["salary"]:
        df[c] = df[c].str.replace(",", "").str.extract(r"^[^\d]*(\d+)").astype(float)

df[domain] = df[[qid for qid in domain_qid_map[domain]]].mean(axis=1)
if domain != "salary":
    df[domain] = df[domain] * 100
df = df.drop(
    columns=[f"q_{i}" for i in range(50)]
    + ["q_59", "q_60"]
    + [f"q_{i}" for i in range(61, 211)],
    errors="ignore",
)
df_linguistic = pd.read_pickle(f"data/{dataset}_utterances_linguistic.gz").drop(
    columns=[
        "s_neutral_model_response",
        "s_neutral_user_prompt",
        "model_response_liwc_Segment",
        "user_prompt_liwc_Segment",
    ],
    errors="ignore",
)
for c in ["politeness_user_prompt", "politeness_model_response"]:
    if c in df_linguistic:
        df_linguistic[c] = df_linguistic[c].replace(
            {"impolite": 0, "neutral": 0.5, "polite": 1, "somewhat polite": 0.75}
        )
df_linguistic = df_linguistic.rename(columns={"gpt_description": "topic"})
all_cols += ["topic"]
group_cols = ["conversation_id"] + all_cols
df_linguistic = (
    df_linguistic.groupby(group_cols)[
        [
            c
            for c in df_linguistic.columns
            if ("model_response" in c or "user_prompt" in c)
            and (c not in ["model_response", "user_prompt"])
        ]
    ]
    .mean()
    .reset_index()
)
df = df.merge(
    df_linguistic[
        ["conversation_id"]
        + [
            c
            for c in df_linguistic.columns
            if "model_response" in c or "user_prompt" in c or c == "topic"
        ]
    ],
    on=["conversation_id"] if dataset != "personamem" else ["conversation_id", "topic"],
)

scaled_df = scale_data(df)
scaled_df[domain] = df[domain]
scaled_df = scaled_df.loc[scaled_df.duplicated(subset=["topic"], keep=False)]
scaled_df = pd.get_dummies(
    scaled_df, columns=all_cols, prefix=["dummy_" + c for c in all_cols]
)
all_cols += [
    c for c in scaled_df.columns if "model_response" in c or "user_prompt" in c
]


filtered_df = scaled_df.loc[~scaled_df[domain].isna()]
filtered_df = filtered_df[
    [c for c in filtered_df.columns if c in all_cols or c.startswith("dummy_")]
    + [domain]
].dropna()

l1_ratios = [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1]  # α candidates
en_model = ElasticNetCV(
    l1_ratio=l1_ratios,
    cv=5,
    max_iter=10000 if dataset != "personamem" else 20000,
    random_state=42,
)
feature_names = [
    c for c in filtered_df.columns if c in all_cols or c.startswith("dummy_")
]

best_model = en_model.fit(
    filtered_df[
        [c for c in filtered_df.columns if c in all_cols or c.startswith("dummy_")]
    ],
    filtered_df[domain],
)

best_alpha = en_model.alpha_
best_l1 = en_model.l1_ratio_
coefs = en_model.coef_
score = best_model.score(
    filtered_df[
        [c for c in filtered_df.columns if c in all_cols or c.startswith("dummy_")]
    ],
    filtered_df[domain],
)

print(f"Best α (alpha) : {best_alpha:.4f}")
print(
    f"Best λ (l1_ratio): {best_l1:.2f}  "
    f"({'pure Lasso' if best_l1==1 else 'pure Ridge' if best_l1==0 else 'Elastic Net'})"
)
print(f"Non-zero coefficients: {np.sum(coefs != 0)} / {len(coefs)}")
print(f"Score: {score}")

# ── 4. Filter to non-zero (selected) features ────────────────────────────────
mask = coefs != 0
sel_names = np.array(feature_names)[mask]
sel_coefs = coefs[mask]

# Sort by absolute value descending
order = np.argsort(np.abs(sel_coefs))[::-1]
sel_names = sel_names[order]
sel_coefs = sel_coefs[order]

sel_names = [s.split("dummy_")[-1][:50] for s in sel_names]

demographic_features = [
    s for s in sel_names if any([d in s for d in demographics[dataset]])
]
dem_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in demographic_features]
mean_dem_feature = np.mean(dem_feature_imp)
max_dem_feature = np.max(dem_feature_imp)

dem_in_20 = sel_names.index(demographic_features[0]) < 20

topic_features = [s for s in sel_names if s.startswith("topic_")]
topic_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in topic_features]
mean_topic_feature = np.mean(topic_feature_imp)
max_topic_feature = np.max(topic_feature_imp)

emotion_features = [s for s in sel_names if s.startswith("e_")]
emotion_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in emotion_features]
mean_emotion_feature = np.mean(emotion_feature_imp)
max_emotion_feature = np.max(emotion_feature_imp)

sent_features = [s for s in sel_names if s.startswith("s_")]
sent_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in sent_features]
mean_sent_feature = np.mean(sent_feature_imp)
max_sent_feature = np.max(sent_feature_imp)

liwc_features = [s for s in sel_names if "_liwc_" in s]
liwc_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in liwc_features]
mean_liwc_feature = np.mean(liwc_feature_imp)
max_liwc_feature = np.max(liwc_feature_imp)

polite_features = [s for s in sel_names if "politeness" in s]
if polite_features:
    polite_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in polite_features]
    mean_polite_feature = np.mean(polite_feature_imp)
    max_polite_feature = np.max(polite_feature_imp)
else:
    mean_polite_feature = 0
    max_polite_feature = 0

concrete_features = [s for s in sel_names if "concreteness" in s]
if concrete_features:
    concrete_feature_imp = [
        abs(sel_coefs[sel_names.index(f)]) for f in concrete_features
    ]
    mean_concrete_feature = np.mean(concrete_feature_imp)
    max_concrete_feature = np.max(concrete_feature_imp)
else:
    mean_concrete_feature = 0
    max_concrete_feature = 0

reading_features = [s for s in sel_names if "flesch_reading_ease" in s]
if reading_features:
    reading_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in reading_features]
    mean_reading_feature = np.mean(reading_feature_imp)
    max_reading_feature = np.max(reading_feature_imp)
else:
    mean_reading_feature = 0
    max_reading_feature = 0

ling_features = [
    s
    for s in sel_names
    if s
    not in reading_features
    + concrete_features
    + polite_features
    + liwc_features
    + sent_features
    + emotion_features
    + topic_features
    + demographic_features
]
ling_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in ling_features]
mean_ling_feature = np.mean(ling_feature_imp)
max_ling_feature = np.max(ling_feature_imp)

user_features = [s for s in sel_names if "user_prompt" in s]
user_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in user_features]
mean_user_feature = np.mean(user_feature_imp)
max_user_feature = np.max(user_feature_imp)

model_features = [s for s in sel_names if "model_response" in s]
model_feature_imp = [abs(sel_coefs[sel_names.index(f)]) for f in model_features]
mean_model_feature = np.mean(model_feature_imp)
max_model_feature = np.max(model_feature_imp)

# ── 5. Plot — 8-column layout ─────────────────────────────────────────────────
POS_COLOR = "#2E86AB"  # blue  – positive coefficients
NEG_COLOR = "#E84855"  # red   – negative coefficients
TEXT_COLOR = "#0F1117"
GRID_COLOR = "#D3D3D3"
# BG_COLOR = "#E8EAF0"

TOP_N = 20
top_names = sel_names[:TOP_N]
top_coefs = sel_coefs[:TOP_N]

fig, ax = plt.subplots(figsize=(14, 6))
# fig.patch.set_facecolor(BG_COLOR)
# ax.set_facecolor(BG_COLOR)

x_pos = np.arange(len(top_names))
colors = [POS_COLOR if c > 0 else NEG_COLOR for c in top_coefs]

bars = ax.bar(x_pos, top_coefs, color=colors, width=0.65, edgecolor="none", zorder=3)

# Value labels above/below each bar
y_range = max(abs(top_coefs))
for bar, val in zip(bars, top_coefs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.03 * y_range * np.sign(val),
        f"{val:+.2f}",
        ha="center",
        va="bottom" if val > 0 else "top",
        fontsize=7.5,
        color=TEXT_COLOR,
        alpha=0.85,
    )

# Rank labels below x-axis tick labels
# for i in range(len(top_names)):
#     ax.text(
#         i, ax.get_ylim()[0] - 0.12 * y_range,
#         f"#{i+1}",
#         ha="center", va="top",
#         fontsize=7, color=TEXT_COLOR, alpha=0.4,
#     )

ax.set_xticks(x_pos)
ax.set_xticklabels(top_names, rotation=35, ha="right", fontsize=9, color=TEXT_COLOR)
ax.tick_params(axis="y", colors=TEXT_COLOR, labelsize=8)
ax.axhline(0, color=TEXT_COLOR, linewidth=0.8, alpha=0.4, zorder=2)
ax.grid(axis="y", color=GRID_COLOR, linewidth=0.6, zorder=1)
ax.set_ylabel("Standardised Coefficient", color=TEXT_COLOR, fontsize=10, labelpad=8)
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)

# Title & subtitle
fig.text(
    0.5,
    0.98,
    "Top 20 Features by Coefficient Magnitude",
    fontsize=13,
    fontweight="bold",
    color=TEXT_COLOR,
    va="top",
    ha="center",
)
fig.text(
    0.5,
    0.945,
    f"λ = {best_alpha:.4f}   |   α (l1_ratio) = {best_l1:.2f}   |   "
    f"{len(sel_coefs)} of {len(en_model.coef_)} features selected",
    fontsize=8.5,
    color=TEXT_COLOR,
    alpha=0.6,
    va="top",
    ha="center",
)

# Legend
pos_patch = mpatches.Patch(color=POS_COLOR, label="Positive effect")
neg_patch = mpatches.Patch(color=NEG_COLOR, label="Negative effect")
ax.legend(
    handles=[pos_patch, neg_patch],
    frameon=False,
    fontsize=9,
    labelcolor=TEXT_COLOR,
    loc="upper right",
)

plt.tight_layout(rect=[0, 0, 1, 0.93])

plt.savefig(
    f"elastic_net_figures/{model}_{dataset}_{domain}.pdf",
    dpi=150,
    bbox_inches="tight",
    # facecolor=BG_COLOR,
)
plt.show()
print(f"Plot saved to elastic_net_figures/{model}_{dataset}_{domain}.pdf")

print(dataset, domain, model)
print(
    "&$"
    + "$& $".join(
        map(
            "{:.2f}".format,
            [
                max_topic_feature,
                mean_topic_feature,
                max_dem_feature,
                mean_dem_feature,
                max_emotion_feature,
                mean_emotion_feature,
                max_polite_feature,
                mean_polite_feature,
                max_sent_feature,
                mean_sent_feature,
                max_concrete_feature,
                mean_concrete_feature,
                max_reading_feature,
                mean_reading_feature,
                max_liwc_feature,
                mean_liwc_feature,
                max_ling_feature,
                mean_ling_feature,
            ],
        )
    )
    + "$\\\\"
)
print(
    "&$"
    + "$& $".join(
        map(
            "{:.2f}".format,
            [
                max_user_feature,
                mean_user_feature,
                max_model_feature,
                mean_model_feature,
            ],
        )
    )
    + "$\\\\"
)
print(dem_in_20)